In [1]:
import duckdb, requests

In [ ]:
manifest = requests.get("https://data.deadlock-api.com/v1/manifest.json").json()
base = manifest["public_url"]

def urls(table: str, limit: int = 50) -> list[str]:
    t = manifest["tables"][table]
    gen = t.get("generation", 0)
    file_list = [f"{base}/{f['key']}" for f in t["files"]
                 if t["policy"] == "snapshot" or f.get("generation", 0) == gen]
    return file_list[-limit:]

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute("SET threads = 2;")
con.execute("SET memory_limit = '4GB';")
con.execute("SET preserve_insertion_order = false;")

file_urls = urls("match_player", limit=30)
con.execute(f"CREATE VIEW match_player AS SELECT * FROM read_parquet({file_urls!r}, union_by_name = true)")

In [6]:
match_player_cols = con.sql("DESCRIBE match_player;").df()["column_name"]
print(match_player_cols)
print(match_player_cols.shape)

0                               match_id
1                             start_time
2                             duration_s
3                             match_mode
4                              game_mode
                     ...                
152    power_up_buffs.pickup_stat_values
153          hero_release_votes.category
154           hero_release_votes.hero_id
155        hero_release_votes.vote_count
156             custom_user_stats_deltas
Name: column_name, Length: 157, dtype: object
(157,)


In [30]:
kd_query = """
SELECT account_id, "stats.kills", kills, "stats.deaths", deaths, start_time
   FROM match_player
   WHERE account_id = 78187819
   ORDER BY start_time DESC
   LIMIT 5;
   """
   
con.execute(kd_query)
print("Testing KD data with Taha account info")
display(con.sql(kd_query).df())

Testing KD data with Taha account info


,account_id,stats.kills,kills,stats.deaths,deaths,start_time
0,78187819,"[0, 0, 0, 0, 0, 1, 1, 2, 2]",2,"[0, 0, 0, 0, 1, 3, 4, 5, 6]",6,2026-10-28 20:51:29-04:00
1,78187819,"[1, 2, 4, 4, 6, 6, 7, 7, 8, 8, 8, 8]",8,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2]",2,2026-10-12 16:58:43-04:00
2,78187819,"[0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 3]",3,"[1, 1, 3, 4, 4, 6, 6, 6, 6, 7, 7, 7]",7,2026-10-11 21:41:36-04:00


In [29]:
stat_query = """
SELECT account_id, "stats.kills", "stats.deaths", "objectives.team_objective", start_time
   FROM match_player
   WHERE account_id = 78187819
   ORDER BY start_time DESC
   LIMIT 5;
   """
   
con.execute(stat_query)
print("Testing objectives compared to kills and death data with Taha account info")
display(con.sql(stat_query).df())

Testing objectives compared to kills and death data with Taha account info


,account_id,stats.kills,stats.deaths,objectives.team_objective,start_time
0,78187819,"[0, 0, 0, 0, 0, 1, 1, 2, 2]","[0, 0, 0, 0, 1, 3, 4, 5, 6]","[Tier1Lane1, Tier1Lane3, Tier1Lane1, Tier1Lane...",2026-10-28 20:51:29-04:00
1,78187819,"[1, 2, 4, 4, 6, 6, 7, 7, 8, 8, 8, 8]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2]","[Tier1Lane1, Tier1Lane3, Tier1Lane4, Tier2Lane...",2026-10-12 16:58:43-04:00
2,78187819,"[0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 3]","[1, 1, 3, 4, 4, 6, 6, 6, 6, 7, 7, 7]","[Tier1Lane3, Tier1Lane4, Tier1Lane1, Tier1Lane...",2026-10-11 21:41:36-04:00


In [28]:
item_count = """
SELECT account_id,
len("stats.kills") as kills_count,
len("stats.deaths") as deaths_count,
len("objectives.team_objective") as team_objective_count,
   start_time
   FROM match_player
   WHERE account_id = 78187819
   ORDER BY start_time DESC
   LIMIT 5;
   """
   
con.execute(item_count)
print("Testing counts of items in lists with Taha account info")
display(con.sql(item_count).df())

Testing counts of items in lists with Taha account info


,account_id,kills_count,deaths_count,team_objective_count,start_time
0,78187819,9,9,20,2026-10-28 20:51:29-04:00
1,78187819,12,12,20,2026-10-12 16:58:43-04:00
2,78187819,12,12,23,2026-10-11 21:41:36-04:00


Kills and deaths seem to match eachother on timelines. Thought they may line up with objective but this is not the case. Because these have an associated time of kill, and time of death it should be possible to use temporally for modelling. I wanted to check this just to see if the data was randomly pulled at different intervals. This does seem to be the case, as deaths jump from 1 to 3 as seen before, but these timeframes most likely don't associate with the objective due to the count not matching.